# Kalman Filter Pairs Trading — XOM vs CVX

**Fixes applied vs original:**
- Fixed variable name confusion: original loaded BA/LMT data but named variables `cvx_data`/`xom_data`
- Added Engle-Granger cointegration test before strategy
- Fixed look-ahead bias: replaced `np.mean(spread)` / `np.std(spread)` (full-period) with rolling z-score
- Fixed cumulative return calculation: `np.cumprod(1 + returns)` instead of `np.cumsum(returns)`
- Fixed annualized return formula (was based on incorrect cumsum total)
- Replaced the broken vectorised + loop position logic with a clean iterative state machine
- Added transaction costs
- Added max drawdown metric

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint, adfuller
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

**Original bug**: cell 0 loaded `gm_with_log.csv` / `f_with_log.csv` (GM / Ford), assigned them to  
`cvx_data` / `xom_data`, then immediately overwrote those variables by loading  
`lmt_with_log.csv` / `ba_with_log.csv` (LMT / BA).  
The Kalman filter then ran on BA/LMT while all axis labels said CVX/XOM.  
Fixed: using the actual XOM and CVX files consistently.

In [ ]:
xom_data = pd.read_csv('xom_with_log.csv')
cvx_data = pd.read_csv('cvx_with_log.csv')

xom_data['Date'] = pd.to_datetime(xom_data['Date'])
cvx_data['Date'] = pd.to_datetime(cvx_data['Date'])

xom_data['Log_Close'] = np.log(xom_data['Close'])
cvx_data['Log_Close'] = np.log(cvx_data['Close'])

# Filter to a fixed window and merge
START, END = '2010-01-01', '2012-12-31'

xom_f = xom_data[(xom_data.Date >= START) & (xom_data.Date <= END)]
cvx_f = cvx_data[(cvx_data.Date >= START) & (cvx_data.Date <= END)]

merged = pd.merge(
    cvx_f[['Date', 'Log_Close']],
    xom_f[['Date', 'Log_Close']],
    on='Date', suffixes=('_CVX', '_XOM')
).sort_values('Date').reset_index(drop=True)

print(f'Observations: {len(merged)}')
print(f'Date range  : {merged.Date.min().date()} to {merged.Date.max().date()}')
merged.head()

## 2. Cointegration Test (Training Period Only)

In [ ]:
y = merged.Log_Close_CVX.values
x = merged.Log_Close_XOM.values

# ADF on individual series (expect non-stationary)
for label, series in [('XOM', x), ('CVX', y)]:
    stat, pval, *_ = adfuller(series, autolag='AIC')
    print(f'ADF {label}: stat={stat:.4f}  p={pval:.4f}  → {"I(1) ✓" if pval > 0.05 else "Stationary — unexpected"}')

print()
t_stat, p_coint, crit_vals = coint(x, y)
print('Engle-Granger Cointegration Test')
print(f'  t-stat  : {t_stat:.4f}')
print(f'  p-value : {p_coint:.4f}')
if p_coint < 0.05:
    print('  → Cointegrated at 5% ✓')
else:
    print('  → WARNING: Not cointegrated — review pair selection')

## 3. Kalman Filter

Models the spread as a linear state-space system:

$$y_t = \beta_t x_t + \alpha_t + \epsilon_t$$

State $[\beta_t, \alpha_t]$ evolves as a random walk.  
The Kalman gain continuously updates the hedge ratio as new data arrives —  
unlike linear regression which produces a single static estimate.

In [ ]:
n = len(y)

# Process noise: separate values for beta and alpha
# beta evolves slowly, alpha can shift more freely
Q = np.diag([1e-5, 1e-4])   # FIX: was scalar Q * np.eye(2), now asymmetric
R = 1e-2                     # Measurement noise

beta_arr  = np.zeros(n)
alpha_arr = np.zeros(n)
P         = np.eye(2) * 1.0  # initial covariance (wide prior)
x_aug     = np.column_stack([x, np.ones(n)])  # [x_t, 1]
state     = np.array([1.0, 0.0])              # initial [beta, alpha]

for t in range(n):
    # Predict
    P_pred = P + Q

    # Update
    obs    = x_aug[t].reshape(1, -1)          # (1, 2)
    S      = obs @ P_pred @ obs.T + R         # innovation covariance (scalar)
    K      = P_pred @ obs.T / S               # Kalman gain (2, 1)
    innov  = y[t] - (obs @ state).item()      # innovation
    state  = state + K.flatten() * innov
    P      = P_pred - K @ obs @ P_pred

    beta_arr[t], alpha_arr[t] = state

# Compute spread
spread = y - (beta_arr * x + alpha_arr)

# Plot Kalman filter estimates
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(beta_arr,  color='blue');   axes[0].set_title('Dynamic Hedge Ratio (β)');  axes[0].set_xlabel('Time')
axes[1].plot(alpha_arr, color='orange'); axes[1].set_title('Dynamic Intercept (α)');    axes[1].set_xlabel('Time')
axes[2].plot(spread,    color='green');  axes[2].set_title('Spread Over Time')
axes[2].axhline(np.mean(spread), color='red', linestyle='--', label='Mean')
axes[2].legend(); axes[2].set_xlabel('Time')

plt.suptitle('Kalman Filter Estimates — XOM / CVX', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Beta  range : [{beta_arr.min():.4f}, {beta_arr.max():.4f}]')
print(f'Alpha range : [{alpha_arr.min():.4f}, {alpha_arr.max():.4f}]')

## 4. Signal Generation with Rolling Z-Score

**Original bug**: signals used `np.mean(spread)` and `np.std(spread)` computed over the *entire* spread array.  
This is look-ahead bias — at time t, you would know the future mean/std of the spread.  
**Fix**: use a rolling window to compute z-scores from past data only.

In [ ]:
ENTRY_Z = 1.5
EXIT_Z  = 0.5
WINDOW  = 30

spread_s = pd.Series(spread)
roll_mean = spread_s.rolling(WINDOW).mean()
roll_std  = spread_s.rolling(WINDOW).std()
z_score   = (spread_s - roll_mean) / roll_std

long_signal  = z_score < -ENTRY_Z   # spread low → expect reversion up → long spread
short_signal = z_score >  ENTRY_Z   # spread high → expect reversion down → short spread

print(f'Long signals  : {long_signal.sum()}')
print(f'Short signals : {short_signal.sum()}')

## 5. Position Tracking (Iterative State Machine)

**Original bug**: vectorised assignment (`positions[long_signal] = 1`) then loop carry-forward  
conflicted — carry-forward condition `positions[i] == 0` was always False on days already assigned  
by the vectorised pass, breaking position persistence.  
**Fix**: single iterative loop with explicit state.

In [ ]:
z_vals   = z_score.values
positions = np.zeros(n)
current   = 0

for i in range(n):
    if np.isnan(z_vals[i]):
        positions[i] = 0
        continue

    if current == 0:
        if z_vals[i] < -ENTRY_Z:    current = 1
        elif z_vals[i] > ENTRY_Z:   current = -1
    elif current == 1  and z_vals[i] > -EXIT_Z:  current = 0
    elif current == -1 and z_vals[i] <  EXIT_Z:  current = 0

    positions[i] = current

print(f'Long  days : {(positions == 1).sum()}')
print(f'Short days : {(positions == -1).sum()}')
print(f'Flat  days : {(positions == 0).sum()}')

## 6. Backtest & Performance Metrics

In [ ]:
TC = 0.0001  # 1 bp round-trip transaction cost

spread_ret = pd.Series(spread).diff().values           # daily spread change
trades     = np.abs(np.diff(positions, prepend=0))     # position changes
strat_ret  = positions[:-1] * spread_ret[1:] - trades[1:] * TC

# FIX: original used np.cumsum(returns) — correct is cumprod(1 + r)
cum_returns = np.cumprod(1 + strat_ret)

# --- Metrics ---
TRADING_DAYS = 252
RF_DAILY     = 0.00001

daily_clean = strat_ret[~np.isnan(strat_ret)]
excess      = daily_clean - RF_DAILY
sharpe      = excess.mean() / excess.std() * np.sqrt(TRADING_DAYS) if excess.std() > 0 else 0

# FIX: annualized return — derived from cumprod, not cumsum
total_return     = cum_returns[-1] - 1
annualized_return = (1 + total_return) ** (TRADING_DAYS / len(strat_ret)) - 1

# Max drawdown
cum_s   = pd.Series(cum_returns)
max_dd  = ((cum_s - cum_s.cummax()) / cum_s.cummax()).min()

# Win rate
nonzero    = daily_clean[daily_clean != 0]
win_rate   = (nonzero > 0).mean() if len(nonzero) > 0 else 0
total_tr   = int(trades.sum())

print('=' * 42)
print('  KALMAN FILTER STRATEGY PERFORMANCE')
print('=' * 42)
print(f'  Cumulative Return  : {total_return*100:>8.2f}%')
print(f'  Annualized Return  : {annualized_return*100:>8.2f}%')
print(f'  Sharpe Ratio       : {sharpe:>8.2f}')
print(f'  Max Drawdown       : {max_dd*100:>8.2f}%')
print(f'  Win Rate           : {win_rate*100:>8.1f}%')
print(f'  Total Trades       : {total_tr:>8d}')

## 7. Visualisations

In [ ]:
dates = merged.Date.values

# Trading signals on log prices
fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

ax = axes[0]
ax.plot(dates, y, label='CVX Log Prices', color='blue', alpha=0.7)
ax.scatter(dates[long_signal.values],  y[long_signal.values],  color='green', marker='^', s=60, zorder=5, label='Long Signal')
ax.scatter(dates[short_signal.values], y[short_signal.values], color='red',   marker='v', s=60, zorder=5, label='Short Signal')
ax.set_title('Trading Signals on CVX Log Prices', fontsize=13)
ax.set_ylabel('Log Price'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(dates, x, label='XOM Log Prices', color='orange', alpha=0.7)
ax.scatter(dates[long_signal.values],  x[long_signal.values],  color='green', marker='^', s=60, zorder=5, label='Long Signal')
ax.scatter(dates[short_signal.values], x[short_signal.values], color='red',   marker='v', s=60, zorder=5, label='Short Signal')
ax.set_title('Trading Signals on XOM Log Prices', fontsize=13)
ax.set_ylabel('Log Price'); ax.set_xlabel('Date'); ax.legend(); ax.grid(True, alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Cumulative returns
plt.figure(figsize=(12, 5))
plt.plot(dates[1:], cum_returns, label='Kalman Filter Strategy', color='green', lw=1.5)
plt.axhline(1.0, color='red', linestyle='--', label='Break-even')
plt.title('Cumulative Returns — Kalman Filter Pairs Trading', fontsize=13)
plt.xlabel('Date'); plt.ylabel('Cumulative Return')
plt.legend(); plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Kalman vs Linear Regression Comparison

| Feature | Linear Regression | Kalman Filter |
|---|---|---|
| Hedge ratio | Static (fixed β) | Dynamic (updates each step) |
| Look-ahead bias risk | Higher (full-period fit) | Lower (recursive updates) |
| Best suited for | Stable long-run relationships | Non-stationary, changing correlations |
| Computational cost | Very low | Low |
| Parameter tuning | Minimal | Q, R matrices |

**Key insight**: For XOM/CVX — two large-cap energy stocks with fundamentally linked prices — the  
static linear regression beta is often sufficient. The Kalman filter adds the most value when the  
pair's relationship shifts over time (e.g., sector rotation, corporate events).